# Following splink's example, for the synthetic dataset

We follow the steps done in 

`https://moj-analytical-services.github.io/splink/demos/examples/duckdb/transactions.html`

while adapting the datasets for the synthetic dataset.

In [1]:
from splink import DuckDBAPI, Linker, SettingsCreator, block_on, splink_datasets
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os

import sys
sys.path.insert(0, os.path.dirname(os.getcwd()))  # Add parent directory

df1_path = os.path.join("..", "..", "output_data", "smallestest_dataset_a.parquet")
df2_path = os.path.join("..", "..", "output_data", "smallestest_dataset_b.parquet")


df1 = pd.read_parquet(df1_path)
df2 = pd.read_parquet(df2_path)

df1['datetime_noisy'] = df1['datetime_noisy'].dt.floor('D')
df2['datetime_noisy'] = df2['datetime_noisy'].dt.floor('D')

display(df1.head(2))
display(df2.head(2))

,global_entity_id,local_source_id,categorical,numerical_noisy,datetime_noisy,latent_event_id
0,100073,DF1_100073,CAT_1,722.39,2025-12-31,OVER_1
1,100073,DF1_100073,CAT_5,655.69,2026-01-01,OVER_0


,global_entity_id,local_source_id,categorical,numerical_noisy,datetime_noisy,latent_event_id
0,100061,DF2_100061,CAT_5,174.62,2025-12-31,UNIQ_B_1332
1,100073,DF1_100073,None,651.94,2025-12-31,OVER_0


In [2]:
print(f"cartesian prod: {len(df1) * len(df2)}")

cartesian prod: 2189808


### Idea: to aggregate the entities first into "behaviours" - or some sort of latent representation that maintains thei information entropy as much as possible

In [3]:
import pandas as pd
import numpy as np

def build_behavioral_profiles(df, id_column):
    profiles = []
    
    # Force datetime cast to access properties cleanly
    df['datetime_noisy'] = pd.to_datetime(df['datetime_noisy'])
    
    for entity_id, group in df.groupby(id_column):
        # 1. Categorical: Extract unique values and join with a comma
        cats = sorted(list(set(group['categorical'].dropna().astype(str))))
        cat_string = ",".join(cats) if cats else None
        
        # 2. Datetime: Map to unique Year-Week tokens to establish historical schedules
        weeks = sorted(list(set(group['datetime_noisy'].dropna().dt.strftime('%G%V'))))
        time_window_string = ",".join(weeks) if weeks else None
        
        # 3. Numerical: Entropy metrics capturing distribution shape
        nums = group['numerical_noisy'].dropna()
        if not nums.empty:
            num_min = float(nums.min())
            num_max = float(nums.max())
            num_median = float(nums.median())
            num_variance = float(nums.var(ddof=0))  # Population variance matching VAR_POP
        else:
            num_min, num_max, num_median, num_variance = None, None, None, None
            
        profiles.append({
            "entity_id": str(entity_id),
            "cat_string": cat_string,
            "time_window_string": time_window_string,
            "num_min": num_min,
            "num_max": num_max,
            "num_median": num_median,
            "num_variance": num_variance
        })
        
    return pd.DataFrame(profiles)

# Form the post-aggregated entity profile dataframes
df1_profiles = build_behavioral_profiles(df1, 'local_source_id')
df2_profiles = build_behavioral_profiles(df2, 'local_source_id')


In [14]:
import pandas as pd
import numpy as np

def build_flat_behavioral_profiles(df, id_column):
    profiles = []
    df['datetime_noisy'] = pd.to_datetime(df['datetime_noisy'])
    
    for entity_id, group in df.groupby(id_column):
        # Flatten distinct string items natively into comma-separated arrays
        cats = sorted(list(set(group['categorical'].dropna().astype(str))))
        cat_string = ",".join(cats) if cats else None
        
        weeks = sorted(list(set(group['datetime_noisy'].dropna().dt.strftime('%G%V'))))
        time_window_string = ",".join(weeks) if weeks else None
        
        nums = group['numerical_noisy'].dropna()
        if not nums.empty:
            num_min = float(nums.min())
            num_max = float(nums.max())
            num_median = float(nums.median())
            num_variance = float(nums.var(ddof=0))
        else:
            num_min, num_max, num_median, num_variance = None, None, None, None
            
        profiles.append({
            "entity_id": str(entity_id),
            "cat_string": cat_string,
            "time_window_string": time_window_string,
            "num_min": num_min,
            "num_max": num_max,
            "num_median": num_median,
            "num_variance": num_variance
        })
        
    return pd.DataFrame(profiles)

# Create clean, flat baseline profile tables
df1_profiles = build_flat_behavioral_profiles(df1, 'local_source_id')
df2_profiles = build_flat_behavioral_profiles(df2, 'local_source_id')

In [17]:
from splink import Linker, SettingsCreator, DuckDBAPI
import splink.comparison_level_library as cll

# Standardised native string array Jaccard template for Splink comparison boxes
# Updated template explicitly using list_length to pass the sqlglot transpile engine cleanly
jaccard_sql_template = """
(
    CASE 
        WHEN {col}_l IS NULL OR {col}_r IS NULL THEN 0.0
        WHEN list_length(string_to_array({col}_l, ',')) = 0 OR list_length(string_to_array({col}_r, ',')) = 0 THEN 0.0
        WHEN list_length(list_intersect(string_to_array({col}_l, ','), string_to_array({col}_r, ','))) = 0 THEN 0.0
        ELSE list_length(list_intersect(string_to_array({col}_l, ','), string_to_array({col}_r, ',')))::DOUBLE / 
             list_length(list_unique(list_concat(string_to_array({col}_l, ','), string_to_array({col}_r, ','))))::DOUBLE
    END
)
"""

# Blocking criteria mapping using the corrected list_length framework
blocking_rule_behavioral = f"""
    {jaccard_sql_template.format(col='cat_string').replace('cat_string_l', 'l.cat_string').replace('cat_string_r', 'r.cat_string')} >= 0.05
"""


# 1. Categorical Behavior Comparison Level
categorical_comparison = {
    "output_column_name": "cat_string",
    "comparison_levels": [
        cll.NullLevel("cat_string"),
        {"sql_condition": f"{jaccard_sql_template.format(col='cat_string')} >= 0.75", "label_for_charts": "High Category Alignment"},
        {"sql_condition": f"{jaccard_sql_template.format(col='cat_string')} >= 0.35", "label_for_charts": "Partial Category Overlap"},
        cll.ElseLevel()
    ]
}

# 2. Datetime Behavior Comparison Level
temporal_comparison = {
    "output_column_name": "time_window_string",
    "comparison_levels": [
        cll.NullLevel("time_window_string"),
        {"sql_condition": f"{jaccard_sql_template.format(col='time_window_string')} >= 0.50", "label_for_charts": "Highly Synchronized Timelines"},
        cll.ElseLevel()
    ]
}

# 3. Numerical Entropy Distribution Levels
numerical_comparison = {
    "output_column_name": "num_median",
    "comparison_levels": [
        cll.NullLevel("num_median"),
        {
            "sql_condition": "ABS(num_median_l - num_median_r) <= 15.0 AND ABS(num_variance_l - num_variance_r) <= 100.0",
            "label_for_charts": "Converged Distribution Medians & Variances"
        },
        cll.ElseLevel()
    ]
}

settings = SettingsCreator(
    link_type="link_only",
    unique_id_column_name="entity_id",
    blocking_rules_to_generate_predictions=[blocking_rule_behavioral],
    comparisons=[categorical_comparison, temporal_comparison, numerical_comparison],
    retain_intermediate_calculation_columns=True
)


In [18]:
import duckdb
con = duckdb.connect(database="splink_temp_workspace.duckdb")
db_api = DuckDBAPI(connection=con)

# Initialize the Linker with raw profiles
linker = Linker([df1_profiles, df2_profiles], settings, db_api=db_api)

# 1. Train the Random Baseline U-parameters
linker.training.estimate_u_using_random_sampling(max_pairs=1e7)

# 2. Train the M-parameters using Expectation-Maximisation (EM)
em_anchor = f"{jaccard_sql_template.format(col='cat_string').replace('cat_string_l', 'l.cat_string').replace('cat_string_r', 'r.cat_string')} >= 0.35"
linker.training.estimate_parameters_using_expectation_maximisation(em_anchor)

# 3. Generate Predictions Matrix
df_predict = linker.inference.predict()


----- Estimating u probabilities using random sampling -----


SplinkException: Error executing the following sql for table `__splink__m_u_counts`(__splink__m_u_counts_51591f7f7):
CREATE TABLE __splink__m_u_counts_51591f7f7 AS
WITH blocked_with_cols AS (
  SELECT
    "l"."source_dataset" AS "source_dataset_l",
    "r"."source_dataset" AS "source_dataset_r",
    "l"."entity_id" AS "entity_id_l",
    "r"."entity_id" AS "entity_id_r",
    "l"."cat_string" AS "cat_string_l",
    "r"."cat_string" AS "cat_string_r",
    "l"."time_window_string" AS "time_window_string_l",
    "r"."time_window_string" AS "time_window_string_r",
    "l"."num_median" AS "num_median_l",
    "r"."num_median" AS "num_median_r",
    "l"."num_variance" AS "num_variance_l",
    "r"."num_variance" AS "num_variance_r",
    b.match_key
  FROM __splink__blocked_id_pairs_8dec4c5d7 AS b
  INNER JOIN __splink__df_concat_sample_4cc340651 AS l
    ON l."source_dataset" || '-__-' || l."entity_id" = b.join_key_l
  INNER JOIN __splink__df_concat_sample_4cc340651 AS r
    ON r."source_dataset

### New attempt

In [4]:
import duckdb

con = duckdb.connect(database="splink_temp_workspace.duckdb")
con.execute("SET memory_limit = '24GB';")
con.execute("SET temp_directory = './duckdb_spill_dir.tmp';")

con.register("raw_df1", df1)
con.register("raw_df2", df2)

# SQL engine to build long-form token representations for both systems
def build_token_representation(source_table, output_view):
    query = f"""
    CREATE OR REPLACE VIEW {output_view} AS
    -- 1. Categorical Tokens
    SELECT DISTINCT 
        local_source_id AS entity_id, 
        'categorical' AS token_type, 
        categorical AS token_val,
        1.0 AS numeric_weight
    FROM {source_table} 
    WHERE categorical IS NOT NULL
    
    UNION ALL
    
    -- 2. Datetime Temporal Schedule Tokens (Year-Weeks)
    SELECT DISTINCT 
        local_source_id AS entity_id, 
        'temporal' AS token_type, 
        strftime(datetime_noisy, '%G%V') AS token_val,
        1.0 AS numeric_weight
    FROM {source_table} 
    WHERE datetime_noisy IS NOT NULL
    
    UNION ALL
    
    -- 3. Numerical Entropy Token: Median
    SELECT 
        local_source_id AS entity_id, 
        'num_median' AS token_type, 
        'median' AS token_val,
        quantile_cont(numerical_noisy, 0.5) AS numeric_weight
    FROM {source_table} 
    GROUP BY local_source_id
    
    UNION ALL
    
    -- 4. Numerical Entropy Token: Variance
    SELECT 
        local_source_id AS entity_id, 
        'num_variance' AS token_type, 
        'variance' AS token_val,
        var_pop(numerical_noisy) AS numeric_weight
    FROM {source_table} 
    GROUP BY local_source_id;
    """
    con.execute(query)

build_token_representation("raw_df1", "df1_tokens")
build_token_representation("raw_df2", "df2_tokens")


In [5]:
con.execute("""
CREATE OR REPLACE TABLE entity_token_sizes AS
SELECT entity_id, token_type, COUNT(*) AS total_tokens
FROM (
    SELECT entity_id, token_type, token_val FROM df1_tokens WHERE token_type IN ('categorical', 'temporal')
    UNION DISTINCT
    SELECT entity_id, token_type, token_val FROM df2_tokens WHERE token_type IN ('categorical', 'temporal')
)
GROUP BY entity_id, token_type;
""")


In [6]:
matrix_generation_query = """
CREATE OR REPLACE TABLE precalculated_behavioral_scores AS
WITH 

-- A. Soft Set Overlap on Categorical Strings
categorical_intersections AS (
    SELECT 
        l.entity_id AS entity_id_l,
        r.entity_id AS entity_id_r,
        COUNT(DISTINCT l.token_val) AS intersection_count
    FROM df1_tokens l
    INNER JOIN df2_tokens r 
        ON l.token_type = 'categorical' AND r.token_type = 'categorical'
        -- Soft Matching Logic: Handles typos, punctuation noise, and spelling variations
        AND jaro_winkler_similarity(l.token_val, r.token_val) >= 0.85
    GROUP BY l.entity_id, r.entity_id
),

-- B. Strict Set Overlap on Temporal Schedules
temporal_intersections AS (
    SELECT 
        l.entity_id AS entity_id_l,
        r.entity_id AS entity_id_r,
        COUNT(*) AS intersection_count
    FROM df1_tokens l
    INNER JOIN df2_tokens r 
        ON l.token_type = 'temporal' AND r.token_type = 'temporal'
        AND l.token_val = r.token_val -- Exact calendar block matches
    GROUP BY l.entity_id, r.entity_id
),

-- C. Numerical Distance Matrix (Medians and Variance differences)
numerical_distances AS (
    SELECT 
        l.entity_id AS entity_id_l,
        r.entity_id AS entity_id_r,
        MAX(CASE WHEN l.token_type = 'num_median' THEN ABS(l.numeric_weight - r.numeric_weight) END) AS median_diff,
        MAX(CASE WHEN l.token_type = 'num_variance' THEN ABS(l.numeric_weight - r.numeric_weight) END) AS variance_diff
    FROM df1_tokens l
    INNER JOIN df2_tokens r 
        ON l.token_type IN ('num_median', 'num_variance') AND r.token_type = l.token_type
    GROUP BY l.entity_id, r.entity_id
),

-- D. Consolidate and build final flat pair matrices
candidate_pairs AS (
    SELECT DISTINCT entity_id_l, entity_id_r FROM categorical_intersections
    UNION DISTINCT
    SELECT DISTINCT entity_id_l, entity_id_r FROM temporal_intersections
)

SELECT 
    p.entity_id_l AS entity_id, -- Splink expects 'entity_id' for left side table tracking
    p.entity_id_r,
    
    -- Compute Soft Jaccard Score
    COALESCE(c.intersection_count::DOUBLE / NULLIF((s1_c.total_tokens + s2_c.total_tokens - c.intersection_count), 0)::DOUBLE, 0.0) AS cat_soft_jaccard,
    
    -- Compute Temporal Schedule Jaccard Score
    COALESCE(t.intersection_count::DOUBLE / NULLIF((s1_t.total_tokens + s2_t.total_tokens - t.intersection_count), 0)::DOUBLE, 0.0) AS temporal_jaccard,
    
    -- Numerical Profile Alignments
    n.median_diff,
    n.variance_diff
    
FROM candidate_pairs p
LEFT JOIN categorical_intersections c ON p.entity_id_l = c.entity_id_l AND p.entity_id_r = c.entity_id_r
LEFT JOIN temporal_intersections t ON p.entity_id_l = t.entity_id_l AND p.entity_id_r = t.entity_id_r
LEFT JOIN numerical_distances n ON p.entity_id_l = n.entity_id_l AND p.entity_id_r = n.entity_id_r
LEFT JOIN entity_token_sizes s1_c ON p.entity_id_l = s1_c.entity_id AND s1_c.token_type = 'categorical'
LEFT JOIN entity_token_sizes s2_c ON p.entity_id_r = s2_c.entity_id AND s2_c.token_type = 'categorical'
LEFT JOIN entity_token_sizes s1_t ON p.entity_id_l = s1_t.entity_id AND s1_t.token_type = 'temporal'
LEFT JOIN entity_token_sizes s2_t ON p.entity_id_r = s2_t.entity_id AND s2_t.token_type = 'temporal';
"""
con.execute(matrix_generation_query)


In [7]:
from splink import Linker, SettingsCreator, DuckDBAPI
import splink.comparison_level_library as cll

# Convert matrix table directly to DataFrames for Splink ingestion layers
df_matching_matrix = con.execute("SELECT * FROM precalculated_behavioral_scores;").df()

# Since we pre-calculated comparisons, we construct two mirror tables to let Splink evaluate pairs
df1_input = df_matching_matrix[['entity_id', 'cat_soft_jaccard', 'temporal_jaccard', 'median_diff', 'variance_diff']].drop_duplicates()
df2_input = df_matching_matrix.rename(columns={'entity_id': 'old_l', 'entity_id_r': 'entity_id'})[['entity_id']].drop_duplicates()

# 1. Soft Jaccard Score Comparison Levels
categorical_comparison = {
    "output_column_name": "cat_soft_jaccard",
    "comparison_levels": [
        cll.NullLevel("cat_soft_jaccard"),
        {"sql_condition": "cat_soft_jaccard_l >= 0.75", "label_for_charts": "High Category Soft Match Overlap"},
        {"sql_condition": "cat_soft_jaccard_l >= 0.35", "label_for_charts": "Partial Category Soft Match Overlap"},
        cll.ElseLevel()
    ]
}

# 2. Schedule Synchronization Comparison Levels
temporal_comparison = {
    "output_column_name": "temporal_jaccard",
    "comparison_levels": [
        cll.NullLevel("temporal_jaccard"),
        {"sql_condition": "temporal_jaccard_l >= 0.60", "label_for_charts": "Highly Synchronized Behavior Timeline"},
        {"sql_condition": "temporal_jaccard_l >= 0.20", "label_for_charts": "Sporadic Overlapping Timeline Activity"},
        cll.ElseLevel()
    ]
}

# 3. Distribution Distance Comparison Levels (Repaired and Closed)
numerical_entropy_comparison = {
    "output_column_name": "median_diff",
    "comparison_levels": [
        cll.NullLevel("median_diff"),
        {
            "sql_condition": "median_diff_l <= 15.0 AND (variance_diff_l IS NULL OR variance_diff_l <= 50.0)",
            "label_for_charts": "Converged Distribution Medians & Variances"
        },
        cll.ElseLevel()
    ],
    "comparison_description": "Absolute differences across precalculated distribution entropy metrics"
}

# Create Settings Object
settings = SettingsCreator(
    link_type="link_only",
    unique_id_column_name="entity_id",
    # Since we pre-filtered the search space in SQL, we can use a basic pass rule
    blocking_rules_to_generate_predictions=["l.cat_soft_jaccard = r.cat_soft_jaccard"], 
    comparisons=[
        categorical_comparison,
        temporal_comparison,
        numerical_entropy_comparison
    ],
    retain_intermediate_calculation_columns=True
)


In [8]:
# Initialize Linker mapping your vertical scoring matrix frames
db_api = DuckDBAPI(connection=con)
linker = Linker([df1_input, df2_input], settings, db_api=db_api)

# 1. Base random pair sampling configuration
linker.training.estimate_u_using_random_sampling(max_pairs=1e6)

# 2. Execute EM Parameter Tuning blocked on significant category overlaps
linker.training.estimate_parameters_using_expectation_maximisation("cat_soft_jaccard_l >= 0.35")

# 3. Generate Final Linkage Score Mapping Output
df_predict = linker.inference.predict()


SETTINGS VALIDATION: Errors were identified in your settings dictionary. 

Invalid Columns(s) in Blocking Rule(s)

    SQL: `l.cat_soft_jaccard = r.cat_soft_jaccard`
       - Missing column(s) from input dataframe(s): `cat_soft_jaccard`

Invalid Columns(s) in Comparison(s)

Comparison: cat_soft_jaccard
--------------------------------------
    SQL: `"cat_soft_jaccard_l" IS NULL OR "cat_soft_jaccard_r" IS NULL`
       - Missing column(s) from input dataframe(s): `cat_soft_jaccard`

    SQL: `cat_soft_jaccard_l >= 0.75`
       - Missing column(s) from input dataframe(s): `cat_soft_jaccard`

    SQL: `cat_soft_jaccard_l >= 0.35`
       - Missing column(s) from input dataframe(s): `cat_soft_jaccard`

Comparison: temporal_jaccard
--------------------------------------
    SQL: `"temporal_jaccard_l" IS NULL OR "temporal_jaccard_r" IS NULL`
       - Missing column(s) from input dataframe(s): `temporal_jaccard`

    SQL: `temporal_jaccard_l >= 0.60`
       - Missing column(s) from input dataf

SplinkException: Error executing the following sql for table `__splink__df_concat_count`(__splink__df_concat_count_d255062dd):
CREATE TABLE __splink__df_concat_count_d255062dd AS
WITH __splink__df_concat AS (
  SELECT
    '__splink__input_table_0' AS source_dataset,
    "entity_id",
    "cat_soft_jaccard",
    "temporal_jaccard",
    "median_diff",
    "variance_diff",
    RAND() AS __splink_salt
  FROM __splink__input_table_0
  UNION ALL
  SELECT
    '__splink__input_table_1' AS source_dataset,
    "entity_id",
    "cat_soft_jaccard",
    "temporal_jaccard",
    "median_diff",
    "variance_diff",
    RAND() AS __splink_salt
  FROM __splink__input_table_1
)
SELECT
  COUNT(source_dataset) AS count
FROM __splink__df_concat
GROUP BY
  source_dataset

Error was: Binder Error: Referenced column "cat_soft_jaccard" not found in FROM clause!
Candidate bindings: "entity_id"

LINE 13:             "entity_id", "cat_soft_jaccard", "temporal_jaccard", "median_diff", ...
                           

In [9]:
import pandas as pd

# Diagnostic Check 1: Review df1 token footprints
df1_tokens_sample = con.execute("SELECT * FROM df1_tokens LIMIT 5;").df()
print("--- [DEBUG STAGE 1] df1_tokens View Sample ---")
print(df1_tokens_sample.to_string())
print(f"Total rows in df1_tokens: {con.execute('SELECT COUNT(*) FROM df1_tokens;').fetchone()[0]}\n")

# Diagnostic Check 2: Review df2 token footprints
df2_tokens_sample = con.execute("SELECT * FROM df2_tokens LIMIT 5;").df()
print("--- [DEBUG STAGE 1] df2_tokens View Sample ---")
print(df2_tokens_sample.to_string())


--- [DEBUG STAGE 1] df1_tokens View Sample ---
    entity_id   token_type token_val  numeric_weight
0  DF1_100087  categorical     CAT_4             1.0
1  DF1_100018  categorical     CAT_6             1.0
2  DF1_100004  categorical     CAT_3             1.0
3  DF1_100025  categorical     CAT_7             1.0
4  DF1_100000  categorical     CAT_3             1.0
Total rows in df1_tokens: 1235

--- [DEBUG STAGE 1] df2_tokens View Sample ---
    entity_id   token_type token_val  numeric_weight
0  DF2_100061  categorical     CAT_1             1.0
1  DF2_100016  categorical     CAT_7             1.0
2  DF2_100034  categorical     CAT_3             1.0
3  DF2_100032  categorical     CAT_2             1.0
4  DF2_100006  categorical     CAT_1             1.0


In [10]:
# Diagnostic Check 3: Read the raw matrix result
matrix_sample = con.execute("SELECT * FROM precalculated_behavioral_scores LIMIT 5;").df()
print("--- [DEBUG STAGE 2] precalculated_behavioral_scores Matrix Sample ---")
print(matrix_sample.to_string())
print(f"Total candidate pairs generated: {len(con.execute('SELECT * FROM precalculated_behavioral_scores;').df())}\n")


--- [DEBUG STAGE 2] precalculated_behavioral_scores Matrix Sample ---
    entity_id entity_id_r  cat_soft_jaccard  temporal_jaccard  median_diff  variance_diff
0  DF1_100088  DF2_100082             0.875          0.428571      202.460  223050.163450
1  DF1_100029  DF2_100049             1.000          0.357143      343.490  695750.832687
2  DF1_100026  DF2_100009             0.875          0.545455      118.790  172783.150445
3  DF1_100074  DF2_100009             0.625          0.368421      384.460  179780.577657
4  DF1_100051  DF2_100009             1.000          0.666667      504.605  100336.154642
Total candidate pairs generated: 2500



In [11]:
# Ingest scores directly from DuckDB
df_matrix = con.execute("SELECT * FROM precalculated_behavioral_scores;").df()

# Structure Input A: Retain left properties, link identifier to right entity
df1_input = df_matrix[[
    'entity_id', 'entity_id_r', 'cat_soft_jaccard', 'temporal_jaccard', 'median_diff', 'variance_diff'
]].copy()

# Structure Input B: Mirror Input A completely so the schema matches exactly
df2_input = df_matrix[[
    'entity_id_r', 'entity_id', 'cat_soft_jaccard', 'temporal_jaccard', 'median_diff', 'variance_diff'
]].copy()
# Rename unique identifiers to make them uniform for Splink ingestion loops
df2_input = df2_input.rename(columns={'entity_id_r': 'entity_id', 'entity_id': 'entity_id_r'})

print("--- [DEBUG STAGE 3] df1_input Schema Validation ---")
print(df1_input.head(2).to_string())
print("\n--- [DEBUG STAGE 3] df2_input Schema Validation ---")
print(df2_input.head(2).to_string())


--- [DEBUG STAGE 3] df1_input Schema Validation ---
    entity_id entity_id_r  cat_soft_jaccard  temporal_jaccard  median_diff  variance_diff
0  DF1_100088  DF2_100082             0.875          0.428571       202.46  223050.163450
1  DF1_100029  DF2_100049             1.000          0.357143       343.49  695750.832687

--- [DEBUG STAGE 3] df2_input Schema Validation ---
    entity_id entity_id_r  cat_soft_jaccard  temporal_jaccard  median_diff  variance_diff
0  DF2_100082  DF1_100088             0.875          0.428571       202.46  223050.163450
1  DF2_100049  DF1_100029             1.000          0.357143       343.49  695750.832687


In [12]:
from splink import Linker, SettingsCreator, DuckDBAPI
import splink.comparison_level_library as cll

# Standardised comparison template checking if precalculated scores match exactly side-by-side
def build_precalculated_comparison(col_name, high_thresh, med_thresh):
    return {
        "output_column_name": col_name,
        "comparison_levels": [
            cll.NullLevel(col_name),
            {
                "sql_condition": f"{col_name}_l >= {high_thresh}",
                "label_for_charts": f"High Overlap (>= {high_thresh})"
            },
            {
                "sql_condition": f"{col_name}_l >= {med_thresh}",
                "label_for_charts": f"Medium Overlap (>= {med_thresh})"
            },
            cll.ElseLevel()
        ]
    }

settings = SettingsCreator(
    link_type="link_only",
    unique_id_column_name="entity_id",
    # CRITICAL BLOCKING: Ensure entity profiles only link with their matching matrix counter-row
    blocking_rules_to_generate_predictions=["l.entity_id_r = r.entity_id_r"],
    comparisons=[
        build_precalculated_comparison("cat_soft_jaccard", 0.75, 0.35),
        build_precalculated_comparison("temporal_jaccard", 0.60, 0.20),
        {
            "output_column_name": "median_diff",
            "comparison_levels": [
                cll.NullLevel("median_diff"),
                {
                    "sql_condition": "median_diff_l <= 15.0 AND (variance_diff_l IS NULL OR variance_diff_l <= 50.0)",
                    "label_for_charts": "Converged Distribution Medians & Variances"
                },
                cll.ElseLevel()
            ]
        }
    ],
    retain_intermediate_calculation_columns=True
)


In [13]:
db_api = DuckDBAPI(connection=con)
linker = Linker([df1_input, df2_input], settings, db_api=db_api)

print("--- [DEBUG STAGE 5] Linker Ingestion Successful! Schema is perfectly aligned. ---")


--- [DEBUG STAGE 5] Linker Ingestion Successful! Schema is perfectly aligned. ---


### Outdated

In [10]:
import duckdb

# Force-clear any conflicting macro versions from your database connection
con.execute("DROP FUNCTION IF EXISTS dynamic_jaccard;")

# Define a robust Python function that handles empty strings, null elements, and calculates Jaccard
def python_jaccard(str1: str, str2: str) -> float:
    if not str1 or not str2:
        return 0.0
    # Clean text components, convert to standard sets
    set1 = set([x.strip() for x in str(str1).split(",") if x.strip()])
    set2 = set([x.strip() for x in str(str2).split(",") if x.strip()])
    
    if not set1 or not set2:
        return 0.0
        
    intersection = len(set1.intersection(set2))
    if intersection == 0:
        return 0.0
        
    union = len(set1.union(set2))
    return float(intersection) / float(union)

# Register the Python function directly inside DuckDB with explicit string conversions
con.create_function("dynamic_jaccard", python_jaccard, [duckdb.typing.VARCHAR, duckdb.typing.VARCHAR], duckdb.typing.DOUBLE)


AttributeError: module 'duckdb' has no attribute 'typing'

In [ ]:
from splink import Linker, SettingsCreator, DuckDBAPI
import splink.comparison_level_library as cll

# Standardised inline DuckDB string-Jaccard calculation query logic
# Replaces macros with explicit, highly performant string splits
jaccard_sql_template = """
(
    CASE 
        WHEN {col}_l IS NULL OR {col}_r IS NULL THEN 0.0
        WHEN len(string_to_array({col}_l, ',')) = 0 OR len(string_to_array({col}_r, ',')) = 0 THEN 0.0
        WHEN len(list_intersect(string_to_array({col}_l, ','), string_to_array({col}_r, ','))) = 0 THEN 0.0
        ELSE len(list_intersect(string_to_array({col}_l, ','), string_to_array({col}_r, ',')))::DOUBLE / 
             len(list_unique(list_concat(string_to_array({col}_l, ','), string_to_array({col}_r, ','))))::DOUBLE
    END
)
"""

# =====================================================================
# CONFIGURING BLOCKING RULES & SETTINGS
# =====================================================================
blocking_rule_behavioral = f"""
    {jaccard_sql_template.format(col='l.cat_string').replace('_l', '')} >= 0.05
    OR {jaccard_sql_template.format(col='l.time_window_string').replace('_l', '')} >= 0.05
"""

blocking_rule_entropy = """
    (l.num_min <= r.num_max AND r.num_min <= l.num_max)
    OR l.num_median IS NULL 
    OR r.num_median IS NULL
"""

brs = [blocking_rule_behavioral, blocking_rule_entropy]

# 1. Categorical Behavior Comparison (Soft String Jaccard)
categorical_behavior_comparison = {
    "output_column_name": "cat_string",
    "comparison_levels": [
        cll.NullLevel("cat_string"),
        {
            "sql_condition": f"{jaccard_sql_template.format(col='cat_string')} >= 0.70",
            "label_for_charts": "High Category Alignment (>=70%)",
        },
        {
            "sql_condition": f"{jaccard_sql_template.format(col='cat_string')} >= 0.30",
            "label_for_charts": "Partial Category Overlap (>=30%)",
        },
        cll.ElseLevel(),
    ],
}

# 2. Datetime Behavior Comparison (Schedule Matching)
temporal_behavior_comparison = {
    "output_column_name": "time_window_string",
    "comparison_levels": [
        cll.NullLevel("time_window_string"),
        {
            "sql_condition": f"{jaccard_sql_template.format(col='time_window_string')} >= 0.50",
            "label_for_charts": "Highly Synchronized Schedules (>=50%)",
        },
        {
            "sql_condition": f"len(list_intersect(string_to_array(time_window_string_l, ','), string_to_array(time_window_string_r, ','))) >= 1",
            "label_for_charts": "Temporal Co-occurrence (At least once)",
        },
        cll.ElseLevel(),
    ],
}

# 3. Numerical Entropy Distribution Comparison
numerical_distribution_comparison = {
    "output_column_name": "num_median",
    "comparison_levels": [
        cll.NullLevel("num_median"),
        {
            "sql_condition": """
                ABS(num_median_l - num_median_r) <= 15 
                AND (
                    (num_variance_l IS NULL OR num_variance_r IS NULL) OR
                    ABS(num_variance_l - num_variance_r) / NULLIF(GREATEST(num_variance_l, num_variance_r), 0) <= 0.25
                )
            """,
            "label_for_charts": "Converged Medians & Similar Variances",
        },
        {
            "sql_condition": "num_min_l <= num_max_r AND num_min_r <= num_max_l",
            "label_for_charts": "Overlapping Distribution Windows",
        },
        cll.ElseLevel(),
    ],
}

settings = SettingsCreator(
    link_type="link_only", 
    unique_id_column_name="entity_id",
    blocking_rules_to_generate_predictions=brs,
    comparisons=[
        categorical_behavior_comparison,
        temporal_behavior_comparison,
        numerical_distribution_comparison
    ],
    retain_intermediate_calculation_columns=True
)


In [25]:
con.execute("DROP FUNCTION IF EXISTS dynamic_jaccard;")

con.execute("""
CREATE OR REPLACE FUNCTION dynamic_jaccard(str1, str2) AS (
    CASE 
        WHEN str1 IS NULL OR str2 IS NULL OR str1 = '' OR str2 = '' THEN 0.0
        ELSE 
            CASE 
                WHEN list_length(list_intersect(string_split(str1, ','), string_split(str2, ','))) = 0 THEN 0.0
                ELSE list_length(list_intersect(string_split(str1, ','), string_split(str2, ',')))::DOUBLE / 
                     list_length(list_unique(list_concat(string_split(str1, ','), string_split(str2, ','))))::DOUBLE
            END
    END
);
""")


In [8]:
from splink import Linker, SettingsCreator, DuckDBAPI
import splink.comparison_level_library as cll

# Standardised inline DuckDB string-Jaccard calculation query logic
# Replaces macros with explicit, highly performant string splits
# 1. Template for Blocking Rules & Training Anchors (Uses l. and r. prefixes)
jaccard_blocking_template = """
(
    CASE 
        WHEN {l_col} IS NULL OR {r_col} IS NULL THEN 0.0
        WHEN len(string_to_array({l_col}, ',')) = 0 OR len(string_to_array({r_col}, ',')) = 0 THEN 0.0
        WHEN len(list_intersect(string_to_array({l_col}, ','), string_to_array({r_col}, ','))) = 0 THEN 0.0
        ELSE len(list_intersect(string_to_array({l_col}, ','), string_to_array({r_col}, ',')))::DOUBLE / 
             len(list_unique(list_concat(string_to_array({l_col}, ','), string_to_array({r_col}, ','))))::DOUBLE
    END
)
"""

# 2. Template for Internal Comparison Levels (Uses _l and _r suffixes)
jaccard_comparison_template = """
(
    CASE 
        WHEN {col}_l IS NULL OR {col}_r IS NULL THEN 0.0
        WHEN len(string_to_array({col}_l, ',')) = 0 OR len(string_to_array({col}_r, ',')) = 0 THEN 0.0
        WHEN len(list_intersect(string_to_array({col}_l, ','), string_to_array({col}_r, ','))) = 0 THEN 0.0
        ELSE len(list_intersect(string_to_array({col}_l, ','), string_to_array({col}_r, ',')))::DOUBLE / 
             len(list_unique(list_concat(string_to_array({col}_l, ','), string_to_array({col}_r, ','))))::DOUBLE
    END
)
"""


# =====================================================================
# CONFIGURING BLOCKING RULES & SETTINGS
# =====================================================================
blocking_rule_behavioral = f"""
    {jaccard_blocking_template.format(l_col='l.cat_string', r_col='r.cat_string')} >= 0.05
    OR {jaccard_blocking_template.format(l_col='l.time_window_string', r_col='r.time_window_string')} >= 0.05
"""

blocking_rule_entropy = """
    (l.num_min <= r.num_max AND r.num_min <= l.num_max)
    OR l.num_median IS NULL 
    OR r.num_median IS NULL
"""

brs = [blocking_rule_behavioral, blocking_rule_entropy]

# 1. Categorical Behavior Comparison
categorical_behavior_comparison = {
    "output_column_name": "cat_string",
    "comparison_levels": [
        cll.NullLevel("cat_string"),
        {
            "sql_condition": f"{jaccard_comparison_template.format(col='cat_string')} >= 0.70",
            "label_for_charts": "High Category Alignment (>=70%)",
        },
        {
            "sql_condition": f"{jaccard_comparison_template.format(col='cat_string')} >= 0.30",
            "label_for_charts": "Partial Category Overlap (>=30%)",
        },
        cll.ElseLevel(),
    ],
}

# 2. Datetime Behavior Comparison
temporal_behavior_comparison = {
    "output_column_name": "time_window_string",
    "comparison_levels": [
        cll.NullLevel("time_window_string"),
        {
            "sql_condition": f"{jaccard_comparison_template.format(col='time_window_string')} >= 0.50",
            "label_for_charts": "Highly Synchronized Schedules (>=50%)",
        },
        {
            "sql_condition": "len(list_intersect(string_to_array(time_window_string_l, ','), string_to_array(time_window_string_r, ','))) >= 1",
            "label_for_charts": "Temporal Co-occurrence (At least once)",
        },
        cll.ElseLevel(),
    ],
}

# 3. Numerical Entropy Distribution Comparison
numerical_distribution_comparison = {
    "output_column_name": "num_median",
    "comparison_levels": [
        cll.NullLevel("num_median"),
        {
            "sql_condition": """
                ABS(num_median_l - num_median_r) <= 15 
                AND (
                    (num_variance_l IS NULL OR num_variance_r IS NULL) OR
                    ABS(num_variance_l - num_variance_r) / NULLIF(GREATEST(num_variance_l, num_variance_r), 0) <= 0.25
                )
            """,
            "label_for_charts": "Converged Medians & Similar Variances",
        },
        {
            "sql_condition": "num_min_l <= num_max_r AND num_min_r <= num_max_l",
            "label_for_charts": "Overlapping Distribution Windows",
        },
        cll.ElseLevel(),
    ],
}

settings = SettingsCreator(
    link_type="link_only", 
    unique_id_column_name="entity_id",
    blocking_rules_to_generate_predictions=brs,
    comparisons=[
        categorical_behavior_comparison,
        temporal_behavior_comparison,
        numerical_distribution_comparison
    ],
    retain_intermediate_calculation_columns=True
)

In [9]:
linker = Linker([df1_profiles, df2_profiles], settings, db_api=db_api)

# 1. Base Random Agreement Adjustments (Now correctly maps l.cat_string vs r.cat_string)
deterministic_anchor = f"{jaccard_blocking_template.format(l_col='l.cat_string', r_col='r.cat_string')} >= 0.50"
linker.training.estimate_probability_two_random_records_match(deterministic_anchor, recall=0.90)

# 2. Random Baseline Sampling
linker.training.estimate_u_using_random_sampling(max_pairs=1e7)

# 3. Execute EM Parameter Tuning Loops
em_block = f"{jaccard_blocking_template.format(l_col='l.cat_string', r_col='r.cat_string')} >= 0.30"
linker.training.estimate_parameters_using_expectation_maximisation(em_block)

# 4. Generate Final Linkage Score Mapping
df_predict = linker.inference.predict()


SplinkException: Error executing the following sql for table `__splink__df_count_cumulative_blocks`(__splink__df_count_cumulative_blocks_a67b4666f):
CREATE TABLE __splink__df_count_cumulative_blocks_a67b4666f AS
WITH __splink__df_concat AS (
  SELECT
    '__splink__input_table_0' AS source_dataset,
    "entity_id",
    "cat_string",
    "time_window_string",
    "num_min",
    "num_max",
    "num_median",
    "num_variance"
  FROM __splink__input_table_0
  UNION ALL
  SELECT
    '__splink__input_table_1' AS source_dataset,
    "entity_id",
    "cat_string",
    "time_window_string",
    "num_min",
    "num_max",
    "num_median",
    "num_variance"
  FROM __splink__input_table_1
), __splink__df_concat_left AS (
  SELECT
    *
  FROM __splink__df_concat
  WHERE
    "source_dataset" = (
      SELECT
        MIN("source_dataset")
      FROM __splink__df_concat
    )
), __splink__df_concat_right AS (
  SELECT
    *
  FROM __splink__df_concat
  WHERE
    "source_dataset" = (
      SELECT
  